# 数据融合预处理：RGB + 光谱指数

**目标**: 
该脚本旨在将`EuroSAT_RGB`数据集的RGB图像与从`EuroSAT_MS`（多光谱）数据集中计算出的光谱指数（NDVI, NDWI, NDBI）进行融合。

**核心流程**:
1.  **加载已有的数据集划分** (`train/valid/test.csv`)。
2.  对每一个样本，同时读取其RGB图像和多光谱`.tif`图像。
3.  计算 **NDVI**, **NDWI**, 和 **NDBI** 指数。
4.  将3个RGB通道和3个指数通道**堆叠（stack）**在一起，形成一个6通道的张量（Tensor）。
5.  将所有通道的尺寸统一调整为 **224x224**，以满足ViT模型的输入要求。
6.  将最终的6通道张量保存为`.pt`文件，存放在一个新的文件夹 (`EuroSAT_fused`) 中，用于后续深度学习模型的训练。

## 1. 导入必要的库

In [1]:
import os
import numpy as np
import torch
import pandas as pd
from PIL import Image
import rasterio
from torchvision.transforms import functional as F
from tqdm.notebook import tqdm

## 2. 路径配置

**请务必检查并确认以下路径与您的项目文件结构完全一致！**

In [2]:
# --- 关键配置区 ---

# 包含所有数据文件夹的根目录
base_data_path = "F:/vscode project/IMA_PW3/data/"

# 各个数据文件夹的路径
data_path_rgb = os.path.join(base_data_path, "EuroSAT_RGB/")
data_path_ms = os.path.join(base_data_path, "EuroSAT_MS/")
metadata_path = os.path.join(base_data_path, "split_info/")

# 创建一个新的文件夹，用于存放融合后的6通道数据
output_path_fused = os.path.join(base_data_path, "EuroSAT_fused/")
os.makedirs(output_path_fused, exist_ok=True)

print(f"RGB图像来源: {data_path_rgb}")
print(f"多光谱图像来源: {data_path_ms}")
print(f"元数据 (CSV) 来源: {metadata_path}")
print(f"融合数据输出至: {output_path_fused}")

RGB图像来源: F:/vscode project/IMA_PW3/data/EuroSAT_RGB/
多光谱图像来源: F:/vscode project/IMA_PW3/data/EuroSAT_MS/
元数据 (CSV) 来源: F:/vscode project/IMA_PW3/data/split_info/
融合数据输出至: F:/vscode project/IMA_PW3/data/EuroSAT_fused/


## 3. 加载数据集划分信息

我们将合并训练、验证和测试集的DataFrame，以便一次性处理所有图片。

In [3]:
try:
    train_df = pd.read_csv(os.path.join(metadata_path, 'train.csv'))
    valid_df = pd.read_csv(os.path.join(metadata_path, 'valid.csv'))
    test_df = pd.read_csv(os.path.join(metadata_path, 'test.csv'))
    all_df = pd.concat([train_df, valid_df, test_df], ignore_index=True)
    print(f"成功加载并合并了 train, valid, test.csv。总计 {len(all_df)} 张图片待处理。")
except FileNotFoundError as e:
    print(f"错误：找不到CSV文件 - {e}。请检查 'metadata_path' 是否正确。")

成功加载并合并了 train, valid, test.csv。总计 27000 张图片待处理。


## 4. 定义光谱指数计算函数

In [4]:
# EuroSAT 数据集的波段信息 (从0开始索引)
# B1: Coastal, B2: Blue, B3: Green, B4: Red
# B5-B7: Red Edge, B8: NIR, B8A: Narrow NIR
# B9: Water Vapour, B10: SWIR-Cirrus, B11: SWIR-1, B12: SWIR-2
BANDS = {
    'GREEN': 2, # 波段 3
    'RED': 3,   # 波段 4
    'NIR': 7,   # 波段 8
    'SWIR1': 10 # 波段 11
}

def calculate_ndvi(image_array):
    """计算归一化植被指数 (NDVI)"""
    nir = image_array[BANDS['NIR'], :, :].astype(float)
    red = image_array[BANDS['RED'], :, :].astype(float)
    denominator = nir + red
    ndvi = np.divide(nir - red, denominator, where=denominator!=0)
    return ndvi

def calculate_ndwi(image_array):
    """计算归一化水体指数 (NDWI)"""
    green = image_array[BANDS['GREEN'], :, :].astype(float)
    nir = image_array[BANDS['NIR'], :, :].astype(float)
    denominator = green + nir
    ndwi = np.divide(green - nir, denominator, where=denominator!=0)
    return ndwi

def calculate_ndbi(image_array):
    """计算归一化建筑指数 (NDBI)"""
    swir1 = image_array[BANDS['SWIR1'], :, :].astype(float)
    nir = image_array[BANDS['NIR'], :, :].astype(float)
    denominator = swir1 + nir
    ndbi = np.divide(swir1 - nir, denominator, where=denominator!=0)
    return ndbi

## 5. 主处理循环：创建并保存6通道数据

此过程会遍历所有图片，执行融合操作并保存。根据您的硬件性能，这可能需要几分钟时间。

In [5]:
# ViT模型通常需要224x224的输入尺寸
TARGET_SIZE = (224, 224)

print(f"开始处理 {len(all_df)} 张图片...")

for index, row in tqdm(all_df.iterrows(), total=len(all_df)):
    try:
        # --- a. 准备文件路径 ---
        img_rel_path_jpg = row['ImagePath']
        img_rel_path_tif = img_rel_path_jpg.replace('.jpg', '.tif')
        rgb_img_path = os.path.join(data_path_rgb, img_rel_path_jpg)
        ms_img_path = os.path.join(data_path_ms, img_rel_path_tif)

        # --- b. 读取并处理RGB图像 ---
        rgb_image = Image.open(rgb_img_path).convert('RGB')
        rgb_tensor = F.to_tensor(rgb_image) # 转换为 [3, H, W] 的Tensor, 值在 [0, 1]

        # --- c. 读取多光谱图像并计算指数 ---
        with rasterio.open(ms_img_path) as src:
            ms_image = src.read()

        ndvi = calculate_ndvi(ms_image)
        ndwi = calculate_ndwi(ms_image)
        ndbi = calculate_ndbi(ms_image)

        # 将指数也转换为Tensor, 并增加一个通道维度 [1, H, W]
        ndvi_tensor = torch.from_numpy(ndvi).unsqueeze(0).float()
        ndwi_tensor = torch.from_numpy(ndwi).unsqueeze(0).float()
        ndbi_tensor = torch.from_numpy(ndbi).unsqueeze(0).float()

        # --- d. 统一所有通道的尺寸 ---
        rgb_tensor_resized = F.resize(rgb_tensor, TARGET_SIZE, antialias=True)
        ndvi_tensor_resized = F.resize(ndvi_tensor, TARGET_SIZE, antialias=True)
        ndwi_tensor_resized = F.resize(ndwi_tensor, TARGET_SIZE, antialias=True)
        ndbi_tensor_resized = F.resize(ndbi_tensor, TARGET_SIZE, antialias=True)

        # --- e. 堆叠通道 ---
        # 将 [3, 224, 224] 和三个 [1, 224, 224] 堆叠成 [6, 224, 224]
        fused_tensor = torch.cat([
            rgb_tensor_resized,
            ndvi_tensor_resized,
            ndwi_tensor_resized,
            ndbi_tensor_resized
        ], dim=0)

        # --- f. 保存为 .pt 文件 ---
        # 确保输出子目录存在 (如 Forest, Pasture)
        output_dir = os.path.join(output_path_fused, os.path.dirname(img_rel_path_jpg))
        os.makedirs(output_dir, exist_ok=True)
        
        # 定义输出文件名，将.jpg替换为.pt
        output_filename = os.path.join(output_path_fused, img_rel_path_jpg.replace('.jpg', '.pt'))
        torch.save(fused_tensor, output_filename)
        
    except Exception as e:
        print(f"\n处理图片 {img_rel_path_jpg} 时发生错误: {e}")

print("\n处理完成！所有6通道数据已保存至 'EuroSAT_fused' 文件夹。")

开始处理 27000 张图片...


  0%|          | 0/27000 [00:00<?, ?it/s]


处理完成！所有6通道数据已保存至 'EuroSAT_fused' 文件夹。


## 6. 验证 (可选)

为了确保一切正常，我们可以随机加载一个刚刚生成的`.pt`文件，并检查其形状（shape）和数据类型（dtype）。

In [6]:
# 随机选择一个样本进行检查
sample_row = all_df.sample(1).iloc[0]
sample_pt_path = os.path.join(output_path_fused, sample_row['ImagePath'].replace('.jpg', '.pt'))

if os.path.exists(sample_pt_path):
    # 加载 Tensor
    loaded_tensor = torch.load(sample_pt_path)
    
    print(f"成功加载样本: {sample_pt_path}")
    print(f"Tensor的形状 (Shape): {loaded_tensor.shape}")
    print(f"Tensor的数据类型 (dtype): {loaded_tensor.dtype}")
    
    # 检查形状是否为 [6, 224, 224]
    if loaded_tensor.shape == (6, 224, 224):
        print("\n验证成功！形状符合预期。")
    else:
        print("\n**警告！** 形状不符合预期，应为 (6, 224, 224)。")
else:
    print(f"错误：找不到验证样本文件 {sample_pt_path}。请检查处理过程是否出错。")

成功加载样本: F:/vscode project/IMA_PW3/data/EuroSAT_fused/HerbaceousVegetation/HerbaceousVegetation_157.pt
Tensor的形状 (Shape): torch.Size([6, 224, 224])
Tensor的数据类型 (dtype): torch.float32

验证成功！形状符合预期。
